# Stage 1b — Smoke Test with a New Base Model: Mana-Persian-Piper
**Project: Persian watermarking on speech synthesized with Mana-Persian-Piper + AudioSeal**

**Why did the base model change?**
Stage 1 was done with Persian MMS-TTS. Watermark detection was perfect (10 out of 10, 100%), but the base speech was almost unintelligible.
Checked with Whisper small:
- MMS heard sentence 1 as «زام حالت چتور هست» and sentence 10 as «خلاصه ساعد نب دانشرا میره وان»
- Mana-Piper heard the same sentences as «سلام و حالت شه توره از» and «فردو سوب ساتن بدانش گاه میره وم» — still not perfect, but much better.

The new model: **Mana-Persian-Piper**, via the `persian-speech` package, trained on the ManaTTS dataset (114 hours, single female speaker).
In an independent evaluation, its ASR character error was 0.057, against 0.14 to 0.26 for older Persian Piper voices.

**Important difference from stage 1:**
- Mana-Piper's output is **22050 Hz**, but AudioSeal works at **16000 Hz**.
- So we first convert each audio file to 16000 with ffmpeg, then watermark it.
- The sentences, the 16-bit message, and the detection method are exactly the same as stage 1, so the numbers stay comparable.

**What do you send back?**
1. The file `summary_step1b.csv`
2. The zip file `persian-watermark-step1b.zip`
3. Answers to the two questions at the end: was Mana better than MMS? Did the watermarked audio sound different from the clean one?


## Cell 1 — Environment check: do we have a GPU?
AudioSeal also works on CPU, but it is faster with a GPU. If there is no GPU, go to the top menu:
select `Runtime > Change runtime type > T4 GPU`, then run this cell again.


In [ ]:
import sys, torch
print("Python:", sys.version)
print("torch:", torch.__version__)
print("Is CUDA available?", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Cell 2 — Install
`persian-speech` downloads the Mana-Persian-Piper model (about 60 MB), `audioseal` installs the watermarking.
It takes a few minutes. Wait for the green tick.


In [ ]:
!pip install -q persian-speech audioseal soundfile 2>&1 | tail -3
print("Install done")


## Cell 3 — Check the install + silence torch warnings
The `torch._dynamo` warnings in stage 1 were just noise and did not affect the result. Here we silence them.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch
try:
    torch._dynamo.config.suppress_errors = True
except Exception as e:
    print("dynamo setting skipped:", e)
import persian_speech_runtime, audioseal
print("persian-speech installed")
print("audioseal installed")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


## Cell 4 — The same 10 fixed Persian sentences
Do not change the sentences. Only the base model changed, so the numbers of stages 1 and 1b can be compared.


In [ ]:
# The 10 fixed Persian test sentences — kept in Persian in every stage so the numbers stay comparable
SENTENCES = [
    "سلام، حالت چطور است؟",
    "هوا امروز در تهران آفتابی است.",
    "من هر روز صبح کتاب می‌خوانم.",
    "هوش مصنوعی دارد زندگی ما را تغییر می‌دهد.",
    "این آزمایش برای بررسی واترمارک فارسی انجام می‌شود.",
    "لطفا در را ببند و چراغ را خاموش کن.",
    "دانشجویان در آزمایشگاه روی پروژه گفتار کار می‌کنند.",
    "صدای ساخته‌شده باید طبیعی و قابل فهم باشد.",
    "ما می‌خواهیم بفهمیم واترمارک روی فارسی هم می‌ماند یا نه.",
    "فردا صبح ساعت نه به دانشگاه می‌روم.",
]
for i, s in enumerate(SENTENCES, 1):
    print(f"{i:02d}. {s}  ({len(s.split())} words)")
print("\nSentence count:", len(SENTENCES))
print("Total words:", sum(len(s.split()) for s in SENTENCES))


## Cell 5 — Load Mana-Persian-Piper
`PiperSpeechEngine()` downloads the `fa_IR-mana-medium.onnx` model on first use.
Its output is 22050 Hz.


In [ ]:
from persian_speech_runtime import PiperSpeechEngine
tts = PiperSpeechEngine()
print("Mana-Persian-Piper ready.")
print("Note: this model's output is 22050 Hz; the next cell converts it to 16000.")


## Cell 6 — Single-sentence test: synthesize only sentence 1
First save the audio at its native rate with `tts.save`, then convert it to 16000 with ffmpeg.
If this one is broken, do not build the rest.


In [ ]:
import os, subprocess, soundfile as sf
from IPython.display import Audio, display

os.makedirs("persian_tts_clean", exist_ok=True)
os.makedirs("tmp_22050", exist_ok=True)

def synth_mana_16k(text, out_16k):
    tmp = out_16k.replace("persian_tts_clean", "tmp_22050").replace("_clean.wav", "_22050.wav")
    tts.save(text, tmp)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp,"-ar","16000",out_16k], check=True)
    wav, sr = sf.read(out_16k)
    assert sr == 16000, f"Sample rate is {sr}, must be 16000"
    return wav, sr

wav1, sr = synth_mana_16k(SENTENCES[0], "persian_tts_clean/utt_001_clean.wav")
print("Sentence 1:", SENTENCES[0])
print("Audio length:", round(len(wav1)/sr, 2), "seconds | samples:", len(wav1), "| rate:", sr)
display(Audio(wav1, rate=sr))
print("Saved: persian_tts_clean/utt_001_clean.wav")


## Cell 7 — Synthesize all 10 clean sentences (16 kHz)
Output: `persian_tts_clean/utt_001_clean.wav` to `utt_010_clean.wav`


In [ ]:
import numpy as np, soundfile as sf, time
durations = []
t0 = time.time()
for i, text in enumerate(SENTENCES, 1):
    wav, sr = synth_mana_16k(text, f"persian_tts_clean/utt_{i:03d}_clean.wav")
    d = len(wav)/sr
    durations.append(d)
    print(f"{i:02d}. {d:.2f}s  persian_tts_clean/utt_{i:03d}_clean.wav")
print(f"\nTotal audio duration: {sum(durations):.2f} seconds")
print(f"Mean per sentence: {np.mean(durations):.2f} seconds")
print(f"Synthesis time (wall-clock): {time.time()-t0:.1f} seconds")


## Cell 8 — Listen: sentences 1, 5 and 10
Compare with stage 1: is «سلام» heard correctly this time? How about sentences 5 and 10?


In [ ]:
from IPython.display import Audio, display
import soundfile as sf
for i in [1, 5, 10]:
    wav, sr = sf.read(f"persian_tts_clean/utt_{i:03d}_clean.wav")
    print(f"--- Sentence {i}: {SENTENCES[i-1]}")
    display(Audio(wav, rate=sr))


## Cell 9 — Load AudioSeal (exactly like stage 1)
- Generator: `audioseal_wm_16bits` — 16-bit message
- Detector: `audioseal_detector_16bits`
- The fixed message is the same as stage 1, so the comparison is fair.


In [ ]:
import torch
from audioseal import AudioSeal
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading AudioSeal generator...")
generator = AudioSeal.load_generator("audioseal_wm_16bits").to(DEVICE)
generator.eval()
print("Loading AudioSeal detector...")
detector = AudioSeal.load_detector("audioseal_detector_16bits").to(DEVICE)
detector.eval()
print("AudioSeal ready.")
MESSAGE = torch.tensor([[1,0,1,1,0,1,0,0,1,1,0,1,0,1,1,0]], dtype=torch.int32, device=DEVICE)
print("Fixed 16-bit message:", MESSAGE.cpu().numpy().tolist()[0])


## Cell 10 — Single-sentence watermark test: only sentence 1
Expected: probability low on clean audio (near 0), high on watermarked audio (near 1).


In [ ]:
import torch, soundfile as sf, numpy as np, os
from IPython.display import Audio, display

def to_tensor(wav_np):
    return torch.from_numpy(wav_np).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

def detect_prob(wav_np):
    wav_t = to_tensor(wav_np)
    with torch.no_grad():
        try:
            result, msg = detector.detect_watermark(wav_t, sample_rate=16000)
        except TypeError:
            result, msg = detector.detect_watermark(wav_t)
    if isinstance(result, torch.Tensor):
        prob = float(result.detach().float().mean().cpu().item())
    else:
        prob = float(result)
    if isinstance(msg, torch.Tensor):
        bits = msg.detach().cpu().numpy().reshape(-1).astype(int).tolist()
    else:
        bits = list(msg)
    return prob, bits

def bit_accuracy(decoded):
    true = MESSAGE.cpu().numpy().reshape(-1).astype(int).tolist()
    correct = sum(int(a)==int(b) for a,b in zip(decoded, true))
    return correct, correct/len(true)

wav_clean, sr = sf.read("persian_tts_clean/utt_001_clean.wav")
p_clean, _ = detect_prob(wav_clean)
print(f"Sentence 1 clean: watermark probability = {p_clean:.3f}  (expected: low)")
with torch.no_grad():
    try:
        wm = generator.get_watermark(to_tensor(wav_clean), sample_rate=16000, message=MESSAGE)
    except TypeError:
        wm = generator.get_watermark(to_tensor(wav_clean), message=MESSAGE)
    wav_wm = (to_tensor(wav_clean) + wm).squeeze().cpu().numpy()
os.makedirs("persian_tts_watermarked", exist_ok=True)
sf.write("persian_tts_watermarked/utt_001_wm.wav", wav_wm, sr)
p_wm, bits_wm = detect_prob(wav_wm)
correct, acc = bit_accuracy(bits_wm)
print(f"Sentence 1 watermarked: probability = {p_wm:.3f}  (expected: high)")
print(f"Decoded bits: {bits_wm}")
print(f"Correct bits: {correct}/16  bit accuracy: {acc:.2%}")
print("\n— Listen: clean first, then watermarked —")
display(Audio(wav_clean, rate=sr))
display(Audio(wav_wm, rate=sr))


## Cell 11 — Watermark all 10 sentences + summary table
Output: `summary_step1b.csv`


In [ ]:
import csv, soundfile as sf
rows = []
for i, text in enumerate(SENTENCES, 1):
    wav_clean, sr = sf.read(f"persian_tts_clean/utt_{i:03d}_clean.wav")
    p_clean, _ = detect_prob(wav_clean)
    with torch.no_grad():
        try:
            wm = generator.get_watermark(to_tensor(wav_clean), sample_rate=16000, message=MESSAGE)
        except TypeError:
            wm = generator.get_watermark(to_tensor(wav_clean), message=MESSAGE)
        wav_wm = (to_tensor(wav_clean) + wm).squeeze().cpu().numpy()
    sf.write(f"persian_tts_watermarked/utt_{i:03d}_wm.wav", wav_wm, sr)
    p_wm, bits_wm = detect_prob(wav_wm)
    correct, acc = bit_accuracy(bits_wm)
    rows.append({"id": i, "text": text, "duration_s": round(len(wav_clean)/sr, 2), "clean_prob": round(p_clean, 3), "wm_prob": round(p_wm, 3), "wm_detected_gt_0.5": bool(p_wm > 0.5), "bits_correct": correct, "bit_accuracy": round(acc, 3)})
    print(f"{i:02d}. clean={p_clean:.3f}  watermarked={p_wm:.3f}  bits={correct}/16")
with open("summary_step1b.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print("\nSaved: summary_step1b.csv")
import pandas as pd
df = pd.DataFrame(rows)
print(df[["id","duration_s","clean_prob","wm_prob","bits_correct"]].to_string(index=False))
print(f"\nMean clean probability: {df['clean_prob'].mean():.3f}")
print(f"Mean watermarked probability: {df['wm_prob'].mean():.3f}")
print(f"Correctly detected (of 10): {int(df['wm_detected_gt_0.5'].sum())}/10")
print(f"Mean bit accuracy: {df['bit_accuracy'].mean():.2%}")
print(f"Total audio duration: {df['duration_s'].sum():.2f} seconds")


## Cell 11b — Stage-1b chart: clean vs watermarked
Drawn from the same `summary_step1b.csv`. Compare it with the stage-1 chart.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
df = pd.read_csv("summary_step1b.csv")
print(df[["id","clean_prob","wm_prob","bits_correct"]].to_string(index=False))
x = np.arange(len(df)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(x - w/2, df["clean_prob"], w, label="Clean")
axes[0].bar(x + w/2, df["wm_prob"], w, label="Watermarked")
axes[0].axhline(0.5, linestyle="--", label="Threshold 0.5")
axes[0].set_xticks(x); axes[0].set_xticklabels(df["id"])
axes[0].set_xlabel("Sentence id"); axes[0].set_ylabel("Detection probability")
axes[0].set_title("Step 1b (Mana-Piper): Clean vs Watermarked detection")
axes[0].set_ylim(0, 1.05); axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
axes[1].bar(df["id"], df["bit_accuracy"])
axes[1].set_xlabel("Sentence id"); axes[1].set_ylabel("Bit accuracy")
axes[1].set_title("Step 1b: Bit accuracy (of 16 bits)")
axes[1].set_ylim(0, 1.05); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("chart_step1b.png", dpi=150); plt.show()
print("\nSaved: chart_step1b.png")
print(f"Mean clean probability: {df['clean_prob'].mean():.3f} | Mean watermarked probability: {df['wm_prob'].mean():.3f}")
print(f"Detected: {int(df['wm_detected_gt_0.5'].sum())}/10 | Mean bit accuracy: {df['bit_accuracy'].mean():.2%}")


## Cell 12 — Small MP3 test (sentence 1 only)
A preview of stage 2: does the watermark survive MP3 at 128 kbps?


In [ ]:
import subprocess, soundfile as sf
subprocess.run(["ffmpeg","-y","-loglevel","error","-i","persian_tts_watermarked/utt_001_wm.wav","-codec:a","libmp3lame","-b:a","128k","/tmp/utt_001_wm.mp3"], check=True)
subprocess.run(["ffmpeg","-y","-loglevel","error","-i","/tmp/utt_001_wm.mp3","/tmp/utt_001_wm_from_mp3.wav"], check=True)
wav_mp3, sr = sf.read("/tmp/utt_001_wm_from_mp3.wav")
print("Sample rate after MP3:", sr)
p_mp3, bits_mp3 = detect_prob(wav_mp3)
correct, acc = bit_accuracy(bits_mp3)
print(f"Sentence 1 after MP3 128k: prob = {p_mp3:.3f} | Correct bits: {correct}/16 ({acc:.0%})")


## Cell 13 — Zip the outputs for download


In [ ]:
import os, zipfile
with zipfile.ZipFile("persian-watermark-step1b.zip", "w") as z:
    for folder in ["persian_tts_clean", "persian_tts_watermarked"]:
        for fn in sorted(os.listdir(folder)):
            z.write(os.path.join(folder, fn))
    z.write("summary_step1b.csv")
    if os.path.exists("chart_step1b.png"):
        z.write("chart_step1b.png")
print("Created: persian-watermark-step1b.zip")
print("Size:", round(os.path.getsize("persian-watermark-step1b.zip")/1024, 1), "KB")
print("Files: 10 clean + 10 watermarked + summary_step1b.csv + chart_step1b.png")


## Done — what do you send back?
**1. Send me the file `summary_step1b.csv`**
**2. Send me the file `persian-watermark-step1b.zip` too**

**And answer these two questions:**
- **A) Listening:** was the Mana voice better than MMS? Did you listen to sentences 1, 5 and 10? Was «سلام» heard correctly this time?
- **B) Listening comparison:** did watermarked sentence 1 sound different from the clean one?

**If stage 1b succeeded, that means:**
- Mean clean probability is low (near 0)
- Mean watermarked probability is high (near 1)
- At least 9 of 10 sentences are detected
- And more importantly: you hear the voice as better than MMS

After this, we build stage 2: robustness against MP3, noise and speed change, for all 10 sentences.
